# 05 — Black-Litterman FinBERT/Momentum

Objectif : tester une version **Black-Litterman** de la stratégie existante.

Au lieu de modifier directement les rendements attendus avec :

```python
mu_adjusted = mu_hist + SIGNAL_STRENGTH * global_score
```

ce notebook transforme les signaux **FinBERT + momentum** en **views Black-Litterman** :

```text
prior neutre / équilibre
+ views issues des global_scores
+ confiance proportionnelle à la force du signal
→ posterior returns
→ PyPortfolioOpt
```

Le notebook ne relance pas FinBERT. Il utilise les CSV générés par `engine_pypfopt.py`.


## 1. Chargement des dépendances et détection du projet

In [ ]:

from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()

if (cwd / "data_output").exists() and (cwd / "results").exists():
    BT_DIR = cwd
elif (cwd / "backtesting_2").exists():
    BT_DIR = cwd / "backtesting_2"
elif cwd.name == "notebooks" and cwd.parent.name == "backtesting_2":
    BT_DIR = cwd.parent
else:
    raise RuntimeError(
        "Impossible de trouver le dossier backtesting_2. "
        "Lance le notebook depuis la racine du repo ou depuis backtesting_2/."
    )

REPO_ROOT = BT_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("BT_DIR     =", BT_DIR)
print("REPO_ROOT  =", REPO_ROOT)


In [ ]:

from backtesting_2.config_backtest import (
    TICKERS,
    INITIAL_CAPITAL,
    WEIGHT_BOUNDS,
    RISK_AVERSION,
    FREQUENCY,
)

DATA_OUTPUT = BT_DIR / "data_output"
RESULTS_DIR = BT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRICES_CSV = DATA_OUTPUT / "prices_backtest.csv"
GLOBAL_SCORES_CSV = DATA_OUTPUT / "daily_global_scores.csv"
PORTFOLIO_RET_CSV = RESULTS_DIR / "portfolio_returns.csv"
WEIGHTS_HIST_CSV = RESULTS_DIR / "weights_history.csv"

required = [PRICES_CSV, GLOBAL_SCORES_CSV, PORTFOLIO_RET_CSV, WEIGHTS_HIST_CSV]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Fichiers manquants :\n" + "\n".join(str(p) for p in missing))

print("Fichiers requis trouvés.")
print("TICKERS =", TICKERS)
print("WEIGHT_BOUNDS =", WEIGHT_BOUNDS)


## 2. Paramètres Black-Litterman

La stratégie utilise des **absolute views** : pour chaque ticker, le `global_score` du jour est converti en rendement attendu annualisé.

Exemple avec `BL_VIEW_SCALE = 0.20` :

```text
global_score = +1.00  → view = +20 % annualisé
global_score = -1.00  → view = -20 % annualisé
global_score =  0.00  → view =   0 % annualisé
```

La confiance dans chaque view dépend de l'intensité du signal :

```text
signal faible  → confiance proche de 35 %
signal fort    → confiance proche de 80 %
```

On clippe le signal entre -1 et +1 pour éviter qu'un z-score extrême domine tout le modèle.


In [ ]:

BL_PRIOR = "equal"        # prior neutre, auto-contenu, sans market caps externes
BL_VIEW_SCALE = 0.20      # même ordre de grandeur que SIGNAL_STRENGTH=0.20
BL_CONFIDENCE_MIN = 0.35  # confiance minimum Idzorek
BL_CONFIDENCE_MAX = 0.80  # confiance maximum Idzorek
BL_TAU = 0.05             # standard PyPortfolioOpt

MIN_EQUITY_EXPOSURE = 0.80  # cash max = 20 %
CASH_DAILY_RETURN = 0.0

params = {
    "BL_PRIOR": BL_PRIOR,
    "BL_VIEW_SCALE": BL_VIEW_SCALE,
    "BL_CONFIDENCE_MIN": BL_CONFIDENCE_MIN,
    "BL_CONFIDENCE_MAX": BL_CONFIDENCE_MAX,
    "BL_TAU": BL_TAU,
    "MIN_EQUITY_EXPOSURE": MIN_EQUITY_EXPOSURE,
}
params


## 3. Chargement des CSV existants

In [ ]:

prices = pd.read_csv(PRICES_CSV, parse_dates=["date"]).set_index("date").sort_index()
prices = prices.reindex(columns=TICKERS).ffill()

global_scores = pd.read_csv(GLOBAL_SCORES_CSV, parse_dates=["date"])
base_returns_df = pd.read_csv(PORTFOLIO_RET_CSV, parse_dates=["date", "next_date"]).sort_values("date")

base_returns = base_returns_df.set_index("date")["portfolio_return"].astype(float)
base_returns.name = "Signal-adjusted PyPortfolioOpt"

overlay_signal = (
    global_scores
    .groupby("date")["global_score"]
    .mean()
    .reindex(base_returns.index)
    .fillna(0.0)
)
overlay_signal.name = "average_global_score"

print("Prix :", prices.index.min().date(), "→", prices.index.max().date(), prices.shape)
print("Backtest :", base_returns.index.min().date(), "→", base_returns.index.max().date(), len(base_returns), "jours")
print("Global scores :", global_scores.shape)

base_returns_df.head()


## 4. Fonctions utilitaires

In [ ]:

def compute_metrics(returns: pd.Series, label: str) -> dict:
    returns = returns.dropna().astype(float)
    n = len(returns)
    total = float((1.0 + returns).prod() - 1.0) if n else 0.0
    ann_ret = float((1.0 + total) ** (252 / n) - 1.0) if n else 0.0
    ann_vol = float(returns.std(ddof=1) * np.sqrt(252)) if n > 1 else 0.0
    sharpe = float((ann_ret - 0.02) / ann_vol) if ann_vol > 1e-10 else 0.0
    cum = (1.0 + returns).cumprod()
    dd = cum / cum.cummax() - 1.0 if n else pd.Series(dtype=float)
    max_dd = float(dd.min()) if n else 0.0
    return {
        "strategie": label,
        "rendement_total": round(total, 4),
        "rendement_annualise": round(ann_ret, 4),
        "volatilite_annualisee": round(ann_vol, 4),
        "sharpe_ratio": round(sharpe, 4),
        "max_drawdown": round(max_dd, 4),
        "n_jours": int(n),
    }


def complete_weights(weights: dict[str, float], tickers: list[str] = TICKERS) -> dict[str, float]:
    completed = {ticker: float(weights.get(ticker, 0.0)) for ticker in tickers}
    total = sum(completed.values())
    if total <= 1e-12:
        return {ticker: 1.0 / len(tickers) for ticker in tickers}
    return {ticker: completed[ticker] / total for ticker in tickers}


def equal_weight(tickers: list[str] = TICKERS) -> dict[str, float]:
    return {ticker: 1.0 / len(tickers) for ticker in tickers}


def day_to_next_returns(prices: pd.DataFrame, schedule_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in schedule_df.iterrows():
        day = row["date"]
        nxt = row["next_date"]
        if day not in prices.index or nxt not in prices.index:
            continue
        rets = (prices.loc[nxt, TICKERS] / prices.loc[day, TICKERS] - 1.0).fillna(0.0)
        rets.name = day
        rows.append(rets)
    return pd.DataFrame(rows)


asset_returns_by_day = day_to_next_returns(prices, base_returns_df)
asset_returns_by_day.head()


## 5. Construction des views Black-Litterman

In [ ]:

def build_bl_views(scores_today: pd.DataFrame, valid_tickers: list[str]) -> tuple[dict[str, float], list[float], pd.Series]:
    """
    Convertit les global_scores du jour en absolute views Black-Litterman.
    """
    scores = (
        scores_today
        .set_index("ticker")["global_score"]
        .reindex(valid_tickers)
        .fillna(0.0)
        .astype(float)
    )

    signal_clipped = scores.clip(-1.0, 1.0)

    views = (BL_VIEW_SCALE * signal_clipped).reindex(valid_tickers)
    absolute_views = {ticker: float(views.loc[ticker]) for ticker in valid_tickers}

    conf = BL_CONFIDENCE_MIN + (BL_CONFIDENCE_MAX - BL_CONFIDENCE_MIN) * signal_clipped.abs()
    conf = conf.clip(0.01, 0.99).reindex(valid_tickers)
    confidences = [float(conf.loc[ticker]) for ticker in valid_tickers]

    return absolute_views, confidences, signal_clipped


first_day = base_returns.index[0]
example_views, example_conf, example_signal = build_bl_views(
    global_scores[global_scores["date"] == first_day],
    TICKERS,
)

pd.DataFrame({
    "signal_clipped": example_signal,
    "absolute_view_annualized": pd.Series(example_views),
    "confidence": pd.Series(example_conf, index=list(example_views.keys())),
})


## 6. Optimiseur Black-Litterman walk-forward

Pour chaque jour J :

```text
prix jusqu'à J uniquement
+ global_scores du jour J
→ posterior returns BL
→ posterior covariance BL
→ EfficientFrontier
→ poids investis de J à J+1
```

Cela évite le look-ahead : le rendement de J à J+1 n'est jamais utilisé pour calculer les poids du jour J.


In [ ]:

def optimize_black_litterman_for_day(
    train_prices: pd.DataFrame,
    scores_today: pd.DataFrame,
) -> tuple[dict[str, float], pd.Series | None, pd.DataFrame | None]:
    """
    Calcule les poids Black-Litterman pour une journée.

    Fallbacks :
    1. Black-Litterman + max_quadratic_utility
    2. Black-Litterman + min_volatility
    3. equal-weight
    """
    from pypfopt.risk_models import CovarianceShrinkage
    from pypfopt.black_litterman import BlackLittermanModel
    from pypfopt.efficient_frontier import EfficientFrontier

    clean_prices = train_prices.reindex(columns=TICKERS).ffill().dropna(how="all")
    clean_prices = clean_prices.dropna(axis=1, how="all")

    valid_tickers = [ticker for ticker in TICKERS if ticker in clean_prices.columns]
    if len(valid_tickers) < 2 or len(clean_prices) < 30:
        return equal_weight(), None, None

    clean_prices = clean_prices.reindex(columns=valid_tickers).dropna(how="any")
    if len(clean_prices) < 30:
        return equal_weight(), None, None

    S = CovarianceShrinkage(clean_prices, frequency=FREQUENCY).ledoit_wolf()
    S = S.reindex(index=valid_tickers, columns=valid_tickers)

    absolute_views, confidences, _ = build_bl_views(scores_today, valid_tickers)

    try:
        bl = BlackLittermanModel(
            cov_matrix=S,
            pi=BL_PRIOR,
            absolute_views=absolute_views,
            omega="idzorek",
            view_confidences=confidences,
            tau=BL_TAU,
            risk_aversion=RISK_AVERSION,
        )

        posterior_rets = bl.bl_returns()
        posterior_cov = bl.bl_cov()

        try:
            ef = EfficientFrontier(posterior_rets, posterior_cov, weight_bounds=WEIGHT_BOUNDS)
            ef.max_quadratic_utility(risk_aversion=RISK_AVERSION)
            weights = dict(ef.clean_weights())
            return complete_weights(weights), posterior_rets, posterior_cov
        except Exception:
            ef = EfficientFrontier(posterior_rets, posterior_cov, weight_bounds=WEIGHT_BOUNDS)
            ef.min_volatility()
            weights = dict(ef.clean_weights())
            return complete_weights(weights), posterior_rets, posterior_cov

    except Exception as exc:
        print(f"Fallback equal-weight BL ({exc})")
        return equal_weight(), None, None


## 7. Backtest Black-Litterman

In [ ]:

bl_return_records = []
bl_weight_records = []
bl_posterior_records = []

portfolio_value = INITIAL_CAPITAL

for i, row in base_returns_df.iterrows():
    day = row["date"]
    next_day = row["next_date"]

    if day not in prices.index or next_day not in prices.index:
        continue

    train_prices = prices.loc[:day, TICKERS]
    scores_today = global_scores[global_scores["date"] == day]

    weights, posterior_rets, _ = optimize_black_litterman_for_day(train_prices, scores_today)

    asset_returns = (prices.loc[next_day, TICKERS] / prices.loc[day, TICKERS] - 1.0).fillna(0.0)
    pf_return = sum(weights.get(ticker, 0.0) * float(asset_returns[ticker]) for ticker in TICKERS)
    portfolio_value *= (1.0 + pf_return)

    bl_return_records.append({
        "date": day,
        "next_date": next_day,
        "portfolio_return": float(pf_return),
        "portfolio_value": float(portfolio_value),
    })

    for ticker in TICKERS:
        bl_weight_records.append({
            "date": day,
            "ticker": ticker,
            "weight": float(weights.get(ticker, 0.0)),
        })

    if posterior_rets is not None:
        for ticker, value in posterior_rets.reindex(TICKERS).items():
            bl_posterior_records.append({
                "date": day,
                "ticker": ticker,
                "posterior_return": float(value) if pd.notna(value) else np.nan,
            })

    if (len(bl_return_records) % 20 == 0) or (i == len(base_returns_df) - 1):
        print(f"{len(bl_return_records):>3} jours traités | {day.date()} | valeur = {portfolio_value:,.2f}")

bl_returns_df = pd.DataFrame(bl_return_records)
bl_weights_df = pd.DataFrame(bl_weight_records)
bl_posterior_df = pd.DataFrame(bl_posterior_records)

bl_returns = bl_returns_df.set_index("date")["portfolio_return"]
bl_returns.name = "Black-Litterman FinBERT/Momentum"
bl_returns_df["cumulative_return"] = (1.0 + bl_returns_df["portfolio_return"]).cumprod() - 1.0

print("Backtest Black-Litterman terminé.")
bl_returns_df.tail()


## 8. Benchmarks et cash overlay

In [ ]:

def apply_cash_overlay(
    returns: pd.Series,
    signal: pd.Series,
    min_equity: float = 0.80,
    low_signal: float = -0.10,
    high_signal: float = 0.05,
    cash_daily_return: float = 0.0,
) -> tuple[pd.Series, pd.Series]:
    """
    Réduit l'exposition actions si le signal moyen est faible.
    """
    signal = signal.reindex(returns.index).fillna(0.0)

    exposure = pd.Series(index=returns.index, dtype=float)
    for date, s in signal.items():
        if s <= low_signal:
            e = min_equity
        elif s >= high_signal:
            e = 1.0
        else:
            x = (s - low_signal) / (high_signal - low_signal)
            e = min_equity + x * (1.0 - min_equity)
        exposure.loc[date] = float(e)

    adjusted = exposure * returns + (1.0 - exposure) * cash_daily_return
    adjusted.name = f"{returns.name} + cash overlay max {int(round((1-min_equity)*100))}%"
    return adjusted, exposure


def compute_equal_weight_returns(asset_returns_by_day: pd.DataFrame) -> pd.Series:
    eq = asset_returns_by_day.reindex(columns=TICKERS).mean(axis=1).fillna(0.0)
    eq.name = "Equal-weight"
    return eq


def compute_classic_static_returns(prices: pd.DataFrame, schedule_df: pd.DataFrame) -> tuple[pd.Series, dict[str, float]]:
    """
    Benchmark PyPortfolioOpt classique : poids calculés une fois au début,
    puis conservés sur toute la période, comme dans analyze_results.py.
    """
    from pypfopt.expected_returns import mean_historical_return
    from pypfopt.risk_models import CovarianceShrinkage
    from pypfopt.efficient_frontier import EfficientFrontier

    first_day = schedule_df["date"].min()
    train = prices.loc[:first_day, TICKERS].ffill().dropna(how="all").dropna(axis=1, how="all")
    valid_tickers = [ticker for ticker in TICKERS if ticker in train.columns]

    if len(train) < 30 or len(valid_tickers) < 2:
        weights = equal_weight()
    else:
        train = train.reindex(columns=valid_tickers).dropna(how="any")
        try:
            mu = mean_historical_return(train, frequency=FREQUENCY)
            S = CovarianceShrinkage(train, frequency=FREQUENCY).ledoit_wolf()
            try:
                ef = EfficientFrontier(mu, S, weight_bounds=WEIGHT_BOUNDS)
                ef.max_quadratic_utility(risk_aversion=RISK_AVERSION)
                weights = complete_weights(dict(ef.clean_weights()))
            except Exception:
                ef = EfficientFrontier(mu, S, weight_bounds=WEIGHT_BOUNDS)
                ef.min_volatility()
                weights = complete_weights(dict(ef.clean_weights()))
        except Exception as exc:
            print(f"Fallback equal-weight benchmark classique ({exc})")
            weights = equal_weight()

    rets = []
    dates = []
    for _, row in schedule_df.iterrows():
        day = row["date"]
        next_day = row["next_date"]
        if day not in prices.index or next_day not in prices.index:
            continue
        asset_rets = (prices.loc[next_day, TICKERS] / prices.loc[day, TICKERS] - 1.0).fillna(0.0)
        pf_ret = sum(weights.get(ticker, 0.0) * float(asset_rets[ticker]) for ticker in TICKERS)
        rets.append(pf_ret)
        dates.append(day)

    classic = pd.Series(rets, index=pd.to_datetime(dates), name="PyPortfolioOpt classique")
    return classic, weights


eq_returns = compute_equal_weight_returns(asset_returns_by_day)
classic_returns, classic_weights = compute_classic_static_returns(prices, base_returns_df)
classic_returns = classic_returns.reindex(base_returns.index).fillna(0.0)

signal_cash20, signal_exp20 = apply_cash_overlay(base_returns, overlay_signal, min_equity=MIN_EQUITY_EXPOSURE)
bl_cash20, bl_exp20 = apply_cash_overlay(bl_returns, overlay_signal, min_equity=MIN_EQUITY_EXPOSURE)
classic_cash20, classic_exp20 = apply_cash_overlay(classic_returns, overlay_signal, min_equity=MIN_EQUITY_EXPOSURE)

print("Poids benchmark classique :", {k: round(v, 4) for k, v in classic_weights.items()})
print("Exposition moyenne signal cash20 :", round(signal_exp20.mean(), 4))
print("Exposition moyenne BL cash20     :", round(bl_exp20.mean(), 4))


## 9. Comparaison des performances

In [ ]:

comparison_series = {
    "Signal-adjusted PyPortfolioOpt": base_returns,
    "Signal-adjusted + cash overlay max 20%": signal_cash20,
    "Black-Litterman FinBERT/Momentum": bl_returns,
    "Black-Litterman + cash overlay max 20%": bl_cash20,
    "PyPortfolioOpt classique": classic_returns,
    "PyPortfolioOpt classique + cash overlay max 20%": classic_cash20,
    "Equal-weight": eq_returns,
}

comparison_series = {
    name: series.reindex(base_returns.index).fillna(0.0).astype(float)
    for name, series in comparison_series.items()
}

comparison = pd.DataFrame([
    compute_metrics(series, name)
    for name, series in comparison_series.items()
])

comparison.sort_values("sharpe_ratio", ascending=False)


## 10. Graphique

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

for name, series in comparison_series.items():
    cumulative_value = (1.0 + series).cumprod() * INITIAL_CAPITAL
    plt.plot(cumulative_value.index, cumulative_value.values, label=name)

plt.axhline(INITIAL_CAPITAL, linewidth=0.8, linestyle="-")
plt.title("Black-Litterman vs stratégies existantes")
plt.ylabel("Valeur du portefeuille ($)")
plt.xlabel("Date")
plt.legend(loc="upper left", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()

chart_path = RESULTS_DIR / "black_litterman_chart.png"
plt.savefig(chart_path, dpi=150)
plt.show()

print("Graphique sauvegardé :", chart_path)


## 11. Exports

In [ ]:

returns_export = pd.DataFrame(comparison_series)
returns_export.index.name = "date"

exposures_export = pd.DataFrame({
    "signal_cash20_exposure": signal_exp20,
    "bl_cash20_exposure": bl_exp20,
    "classic_cash20_exposure": classic_exp20,
})
exposures_export.index.name = "date"

comparison_path = RESULTS_DIR / "black_litterman_comparison.csv"
returns_path = RESULTS_DIR / "black_litterman_returns.csv"
weights_path = RESULTS_DIR / "black_litterman_weights.csv"
posterior_path = RESULTS_DIR / "black_litterman_posterior_returns.csv"
exposures_path = RESULTS_DIR / "black_litterman_cash_exposures.csv"
summary_path = RESULTS_DIR / "black_litterman_summary.json"

comparison.to_csv(comparison_path, index=False)
returns_export.to_csv(returns_path)
bl_weights_df.to_csv(weights_path, index=False)
bl_posterior_df.to_csv(posterior_path, index=False)
exposures_export.to_csv(exposures_path)

summary = {
    "params": params,
    "classic_weights": classic_weights,
    "metrics": comparison.to_dict(orient="records"),
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("Exports créés :")
for p in [comparison_path, returns_path, weights_path, posterior_path, exposures_path, summary_path, chart_path]:
    print("-", p)


## 12. Diagnostic des poids Black-Litterman

In [ ]:

weights_pivot = bl_weights_df.pivot(index="date", columns="ticker", values="weight").fillna(0.0)

weight_stats = pd.DataFrame({
    "mean_weight": weights_pivot.mean(),
    "min_weight": weights_pivot.min(),
    "max_weight": weights_pivot.max(),
    "days_at_upper_bound": (weights_pivot >= WEIGHT_BOUNDS[1] - 1e-4).sum(),
    "days_at_lower_bound": (weights_pivot <= WEIGHT_BOUNDS[0] + 1e-4).sum(),
})

weight_stats.sort_values("mean_weight", ascending=False)


## 13. Conclusion automatique

In [ ]:

best = comparison.sort_values("sharpe_ratio", ascending=False).iloc[0]
bl_row = comparison[comparison["strategie"] == "Black-Litterman FinBERT/Momentum"].iloc[0]
bl_cash_row = comparison[comparison["strategie"] == "Black-Litterman + cash overlay max 20%"].iloc[0]
signal_cash_row = comparison[comparison["strategie"] == "Signal-adjusted + cash overlay max 20%"].iloc[0]

print("Meilleure stratégie au Sharpe :", best["strategie"])
print()
print("Black-Litterman sans cash :")
print(bl_row.to_string())
print()
print("Black-Litterman avec cash overlay 20 % :")
print(bl_cash_row.to_string())
print()
print("Signal-adjusted avec cash overlay 20 % :")
print(signal_cash_row.to_string())
print()

if bl_cash_row["sharpe_ratio"] > signal_cash_row["sharpe_ratio"]:
    print("Conclusion provisoire : Black-Litterman + cash overlay améliore la stratégie signal-adjusted + cash overlay.")
else:
    print("Conclusion provisoire : Black-Litterman n'améliore pas encore la stratégie signal-adjusted + cash overlay.")
    print("À tester ensuite : VIEW_SCALE, bornes de confiance, ou views relatives top-minus-bottom.")
